In [ ]:
import subprocess, sys, time, statistics, pandas as pd
from pathlib import Path

N_REPEATS = 20

# ------------------------------------------------------------------
c_values = [0.005, 0.01, 0.005, 0.001, 0.001, 0.005,  0.005, 0.001]
data= ["Cornell","Dermatology","Glass","HHAR","ISOLET","ORL","USPS","Vehicle"]
# ------------------------------------------------------------------
EPSILONS   = [1.0, 2.0, 4.0, 8.0]
N_REPEATS  = 20
BASE_SEED  = 42                   
OUT_DIR    = Path("exp_results_sklearn"); OUT_DIR.mkdir(exist_ok=True)

# ────────────────────────────────────────────────────────────────────────────
def run_once(ds, dp_method, C, eps, seed):
    multi_class = "crammer_singer" if dp_method == "pmsvm" else "ovr"
    cmd = [
        "python", "main-sklearn.py",
        "--data", ds,
        "--optimizer", "dp_svc",
        "--C", str(C),
        "--eps", str(eps),
        "--dp_method", dp_method,
        "--multi_class", multi_class,
        "--state", str(seed),              
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    out, err = proc.communicate()
    if err: print(err, file=sys.stderr)

    acc = None
    for ln in out.splitlines():
        low = ln.lower()
        if "final test accuracy" in low or "test accuracy" in low:
            nums = [s for s in low.replace('%','').split() if s.replace('.','',1).isdigit()]
            if nums:
                acc = float(nums[-1]); break
    if acc is None:
        raise RuntimeError(f"accuracy not found for {ds}-{dp_method}, eps={eps}")
    return acc
# ────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    rows = []
    for ds, C in zip(data,c_values):
        for dp_method in ["privatesvm", "opera", "pmsvm"]:
            best_C   = None
            best_mu  = -1.0   
            best_std = 0.0

            acc_list = []
            for eps in EPSILONS:
                tmp_acc = []
                for rep in range(N_REPEATS):
                    seed = BASE_SEED + rep
                    acc  = run_once(ds, dp_method, C, eps, seed)
                    tmp_acc.append(acc)
                    time.sleep(0.2)
                mu, sd = statistics.mean(tmp_acc), statistics.stdev(tmp_acc)
                acc_list.append((eps, mu, sd))
                print(f"[{ds:10s} | {dp_method:11s} | C={C:.3g} | ε={eps}] {mu:.3f}±{sd:.3f}")

            overall_mu = statistics.mean([x[1] for x in acc_list])
            if overall_mu > best_mu:
                best_mu, best_std, best_C = overall_mu, sd, C
                best_acc_list = acc_list   

            for eps, mu, sd in best_acc_list:
                rows.append({
                    "dataset": ds,
                    "dp_method": dp_method,
                    "epsilon": eps,
                    "C": best_C,
                    "mean": mu,
                    "std": sd,
                    "latex": f"{mu:.3f}$\\pm${sd:.3f}"
                })

            print(f"[{ds} | {dp_method}] selected C={best_C}")

            pd.DataFrame(rows).to_csv(OUT_DIR/"summary_progress.csv", index=False)
